# 02 - Data Preprocessing
## Fake News Detection - Text Cleaning and Preprocessing

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('stopwords')
nltk.download('punkt')

# Load data
df = pd.read_csv('../data/WELFake_Dataset.csv')
print(f"Original shape: {df.shape}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Original shape: (72134, 4)


In [2]:
# Handle missing values
print("Missing values before:")
print(df.isnull().sum())

df = df.dropna(subset=['text'])
print(f"\nShape after removing missing text: {df.shape}")

Missing values before:
Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

Shape after removing missing text: (72095, 4)


In [3]:
# Text preprocessing function
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning
df['cleaned_text'] = df['text'].apply(clean_text)

# Remove empty texts
df = df[df['cleaned_text'].str.len() > 0]
print(f"Shape after cleaning: {df.shape}")

Shape after cleaning: (71222, 5)


In [4]:
# Tokenization and stopword removal
stop_words = set(stopwords.words('english'))

def tokenize_and_remove_stopwords(text):
    tokens = text.split()
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

df['tokens'] = df['cleaned_text'].apply(tokenize_and_remove_stopwords)
print(f"Sample tokens: {df['tokens'].iloc[0][:20]}")

Sample tokens: ['comment', 'expected', 'barack', 'obama', 'members', 'fyf', 'fukyoflag', 'blacklivesmatter', 'movements', 'called', 'lynching', 'hanging', 'white', 'people', 'cops', 'encouraged', 'others', 'radio', 'show', 'tuesday']


In [5]:
# Stemming
stemmer = PorterStemmer()

def apply_stemming(tokens):
    return [stemmer.stem(token) for token in tokens]

df['stemmed_tokens'] = df['tokens'].apply(apply_stemming)

# Reconstruct text
df['processed_text'] = df['stemmed_tokens'].apply(lambda x: ' '.join(x))
print(f"Sample processed text: {df['processed_text'].iloc[0][:200]}")

Sample processed text: comment expect barack obama member fyf fukyoflag blacklivesmatt movement call lynch hang white peopl cop encourag other radio show tuesday night turn tide kill white peopl cop send messag kill black p


In [6]:
# Compare original vs processed
sample_idx = 0
print("ORIGINAL TEXT:")
print(df['text'].iloc[sample_idx][:500] + "...")
print("\nPROCESSED TEXT:")
print(df['processed_text'].iloc[sample_idx][:500] + "...")

ORIGINAL TEXT:
No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about the killing of black people in America.One of the F***YoFlag organizers is called  Sunshine.  She has a radio blog show hosted from Texas called,  Sunshine s F***ing Opinion Radio Show. A snapshot of h...

PROCESSED TEXT:
comment expect barack obama member fyf fukyoflag blacklivesmatt movement call lynch hang white peopl cop encourag other radio show tuesday night turn tide kill white peopl cop send messag kill black peopl americaon fyoflag organ call sunshin radio blog show host texa call sunshin fing opinion radio show snapshot fyf lolatwhitefear twitter page pm show urg support call fyf tonight continu dismantl illus white snapshot twitter radio call invit fyfth radio show a

In [7]:
# Add processed text length columns
df['processed_length'] = df['processed_text'].str.len()
df['processed_words'] = df['processed_text'].str.split().str.len()

# Display summary
print("\nProcessed Text Statistics:")
print(f"Average length: {df['processed_length'].mean():.0f} characters")
print(f"Average words: {df['processed_words'].mean():.0f} words")
print(f"Max words: {df['processed_words'].max()}")
print(f"Min words: {df['processed_words'].min()}")


Processed Text Statistics:
Average length: 1986 characters
Average words: 305 words
Max words: 12127
Min words: 0


In [8]:
# Save preprocessed data
df.to_csv('../data/preprocessed_news.csv', index=False)
print("✅ Preprocessed data saved to '../data/preprocessed_news.csv'")

✅ Preprocessed data saved to '../data/preprocessed_news.csv'
